In [3]:
import os
from pathlib import Path

DATA_ROOT = Path("Data")
from read.aip import AIPParser  # ✅ 确保有 read/aip.py 且内有 AIPParser


def parse_aip_article(filepath, output_folder):
    """解析单篇 AIP HTML 文件并将 paragraph 写入 txt"""
    try:
        # 1. 初始化解析器
        aip_parser = AIPParser(filepath)

        # 2. 解析元信息
        meta_data = aip_parser.parse_meta()  # 依赖上面我们让 parse_meta 返回 dict
        print(f"📄 Title: {meta_data.get('title', 'N/A')}")
        print(f"📘 Journal: {meta_data.get('journal', 'N/A')}")
        # print(f"🧾 Abstract: {meta_data.get('abstract', 'N/A')}")
        # print(f"📅 Date: {meta_data.get('date', 'N/A')}")

        # 3. 解析段落（AIPParser.parse_paragraphs 返回的是元素列表）
        para_elements = aip_parser.parse_paragraphs()
        paragraph_texts = []

        for el in para_elements:
            # lxml 元素：有 text_content 方法
            if hasattr(el, "text_content"):
                txt = el.text_content().strip()
            else:
                txt = str(el).strip()

            if txt:
                paragraph_texts.append(txt)

        print(f"📝 段落数: {len(paragraph_texts)}")

        # 4. 构建输出文件名
        base_name = os.path.basename(filepath)
        # 去掉 .html / .htm 等后缀
        for ext in [".html", ".htm", ".xhtml"]:
            if base_name.lower().endswith(ext):
                base_name = base_name[: -len(ext)]
                break

        safe_name = "".join(c for c in base_name if c.isalnum() or c in (" ", "_", "-"))
        output_path = os.path.join(output_folder, f"{safe_name}.txt")

        # 5. 写入到 txt 文件
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(f"Title: {meta_data.get('title', '')}\n")
            f.write(f"Journal: {meta_data.get('journal', '')}\n")
            f.write(f"Date: {meta_data.get('date', '')}\n")
            f.write(f"Abstract: {meta_data.get('abstract', '')}\n\n")
            f.write("Paragraphs:\n")
            for para in paragraph_texts:
                f.write(para + "\n\n")

        print(f"✅ 已保存段落 → {output_path}\n")

    except Exception as e:
        print(f"❌ 解析失败：{filepath}\n错误信息：{e}\n")


def batch_parse_aip_folder(input_folder, output_folder):
    """批量解析 AIP HTML 文件，并跳过已生成的 TXT"""
    os.makedirs(output_folder, exist_ok=True)

    html_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith((".html", ".htm"))
    ]

    if not html_files:
        print("⚠️ 未找到 HTML 文件，请检查路径。")
        return

    print(f"🚀 开始批量解析 AIP 文献，共 {len(html_files)} 篇...\n")

    for html_file in html_files:
        # --- 1. 预先构建目标 TXT 的文件名 (需与 parse_aip_article 内部逻辑一致) ---
        base_name = html_file
        for ext in [".html", ".htm"]:
            if base_name.lower().endswith(ext):
                base_name = base_name[: -len(ext)]
                break
        
        safe_name = "".join(c for c in base_name if c.isalnum() or c in (" ", "_", "-"))
        output_path = os.path.join(output_folder, f"{safe_name}.txt")

        # --- 2. 核心判断：如果文件已存在且不是空文件，则跳过 ---
        if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            # print(f"⏭️  跳过已存在文件: {safe_name}.txt")
            continue
        # ------------------------------------------------------------------

        file_path = os.path.join(input_folder, html_file)
        parse_aip_article(file_path, output_folder)

    print("🎯 全部解析完成！")


if __name__ == "__main__":
    # 👉 换成你的 AIP HTML 文件夹路径
    input_folder = DATA_ROOT / "AIP" / "source"     # 📂 输入 HTML 文件夹路径
    output_folder = DATA_ROOT / "AIP" / "txt"       # 📂 输出 TXT 文件夹路径
    batch_parse_aip_folder(input_folder, output_folder)


🚀 开始批量解析 AIP 文献，共 372 篇...


🧩 --- META INFO ---
[DEBUG] Primary XPath hits:
  title: 1 nodes
  journal: 0 nodes
  abstract: 0 nodes
  date: 0 nodes
📄 Title: Nitridation behavior of sapphire using a carbon-saturated N2–CO gas mixture Available
📘 Journal: Journal of Applied Physics
✅ Found 65 paragraphs.

📝 段落数: 65
✅ 已保存段落 → D:\FXR\1111-HTML\AIP\AIP-TXT\101063_13272692.txt


🧩 --- META INFO ---
[DEBUG] Primary XPath hits:
  title: 1 nodes
  journal: 0 nodes
  abstract: 0 nodes
  date: 0 nodes
📄 Title: Modeling nonlinear electromechanical behavior of shocked silicon carbide Available
📘 Journal: Journal of Applied Physics
✅ Found 181 paragraphs.

📝 段落数: 181
✅ 已保存段落 → D:\FXR\1111-HTML\AIP\AIP-TXT\101063_13277030.txt


🧩 --- META INFO ---
[DEBUG] Primary XPath hits:
  title: 1 nodes
  journal: 0 nodes
  abstract: 0 nodes
  date: 0 nodes
📄 Title: Modified spontaneous emission rate in three-dimensional layer-by-layer photonic crystals with planar defects Available
📘 Journal: Journal of Applie